In [1]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)

# 11 报告模块 (report)

提供专业的模型报告生成功能。

**数据说明**: 基于 `hscredit_yyp.xlsx`

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path

from hscredit import init_setting
from hscredit.report import ExcelWriter, feature_bin_stats, FeatureAnalyzer

init_setting()

# 加载数据
_roots = [Path.cwd(), Path.cwd() / 'examples', Path.cwd().parent]
_fp = None
for _r in _roots:
    for _n in ('hscredit_yyp.xlsx', 'hengshucredit_yyp.xlsx'):
        if (_r / _n).is_file():
            _fp = _r / _n
            break
    if _fp is not None:
        break
if _fp is None:
    raise FileNotFoundError('请将 hscredit_yyp.xlsx 放在 examples/')

df = pd.read_excel(_fp)

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

print(f"样本数: {len(df):,}")
print(f"坏样本率: {df['target'].mean():.2%}")

样本数: 970
坏样本率: 16.70%


## 1. Excel报告生成

In [ ]:
# 使用ExcelWriter创建报告
with ExcelWriter().set_filename('demo_report.xlsx') as writer:
    # 写入数据概览
    summary_df = pd.DataFrame({
        '指标': ['样本数', '特征数', '坏样本数', '坏样本率', '好样本数'],
        '值': [len(df), df.shape[1], df['target'].sum(), 
               f"{df['target'].mean():.2%}", len(df) - df['target'].sum()]
    })
    ws1 = writer.get_sheet_by_name('数据概览')
    writer.insert_df2sheet(ws1, summary_df, "B2")
    
    # 写入特征统计
    stats_df = df.describe().reset_index()
    ws2 = writer.get_sheet_by_name('特征统计')
    writer.insert_df2sheet(ws2, stats_df, "B2")

print("Excel报告已生成: demo_report.xlsx")

Excel报告已生成: demo_report.xlsx


## 2. 特征分箱统计

In [ ]:
# 特征分箱统计
feature_name = '中智小牛分C3'

stats = feature_bin_stats(
    data=df,
    feature=feature_name,
    target='target',
    max_n_bins=10
)

print(f"=== {feature_name} 分箱统计 ===")
display(stats.head(10))

TypeError: feature_bin_stats() missing 1 required positional argument: 'data'

## 3. 特征分析报告

In [ ]:
# 使用FeatureAnalyzer进行特征分析
analyzer = FeatureAnalyzer()

features_to_analyze = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1']

result = analyzer.analyze(
    data=df,
    feature=features_to_analyze,
    target='target'
)

display(result.head(20))

# 导出到 Excel
from hscredit.report import dataframe2excel

dataframe2excel(result, 'feature_analysis_report.xlsx', sheet_name='特征分箱统计')
print("特征分析报告已生成: feature_analysis_report.xlsx")

TypeError: FeatureAnalyzer.analyze() missing 1 required positional argument: 'data'